In [1]:
#to create a web server in Python

In [2]:
pip install fastapi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/") #?? why is there a slash inside here and /submit in post, what else we could do here??
async def get_ex():
    return{"message":"Hello world "}

@app.post("/submit") # /submit is the URL path that tells client which action to perform
async def post_ex():
    return {"message":"posted"}

# More actions are:
#@app.post("/predict") 
#@app.post("/v1/chat/completions")


# 2. Dynamic Path Parameters (variables inside the URL)
@app.get("/users/{user_id}")
async def get_user(user_id: int):
    return {user_id: user_id}

@app.get("/models/{model_name}/status")
async def model_status(model_name: str):
    return {"model": model_name, "status": "ready"}

@app.get("/models/{model_name}")
async def get_model_info(model_name: str):
    return {
        "model_requested": model_name,
        "is_supported": model_name in ["gpt-4o", "claude-3-5-sonnet", "llama-3"]
    }

In [11]:
client = TestClient(app)
print("Server will return **HTTP Response object**. when using cliend.get(), client.post()")
res_get = client.get("/")
print("GET Result:", res_get.status_code, res_get.json())

res_mismatch = client.post("/")
print("Mismatch result:", res_mismatch.status_code, res_mismatch.json())

# 'json = ' is used to convert py dict to json string and alerts the server that it is sending JSON data.
res_post = client.post("/submit", json ={"user" : "dev_user"}) 
print("Post result", res_post.status_code, res_post.json())

print(client.get("/models/random-model").json())
print(client.get("/models/gpt-4o").json())


Server will return **HTTP Response object**. when using cliend.get(), client.post()
GET Result: 200 {'message': 'Hello world '}
Mismatch result: 405 {'detail': 'Method Not Allowed'}
Post result 200 {'message': 'posted'}
{'model_requested': 'random-model', 'is_supported': False}
{'model_requested': 'gpt-4o', 'is_supported': True}


In [12]:
# It looks like the line breaks got stripped during the paste, turning multiple statements into one long line!

# Here i/s the clean snippet formatted with explicit newlines. You can copy this directly into your Jupyter cell:
# 
# ```python
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI()

# 1. Define schema
class UserProfile(BaseModel):
    username: str
    tier: str = "free"
    token_limit: int = Field(gt=0)

# 2. Endpoint with validation
@app.post("/register")
async def register_user(profile: UserProfile):
    return {
        "status": "success",
        "saved_user": profile.username,
        "tokens": profile.token_limit
    }

# 3. Test client
client = TestClient(app)

# Test 1: Valid input (200 OK)
res_valid = client.post(
    "/register", 
    json={"username": "alex", "token_limit": 5000}
)
print("Test 1 (Valid):", res_valid.status_code, res_valid.json())

# Test 2: Invalid input (token_limit <= 0 triggers 422)
res_invalid = client.post(
    "/register", 
    json={"username": "alex", "token_limit": -10}
)
print("Test 2 (Invalid):", res_invalid.status_code, res_invalid.json())


# Give this a run—you should see Test 1 succeed with `200` and Test 2 fail cleanly with `422 Unprocessable Entity`!

Test 1 (Valid): 200 {'status': 'success', 'saved_user': 'alex', 'tokens': 5000}
Test 2 (Invalid): 422 {'detail': [{'type': 'greater_than', 'loc': ['body', 'token_limit'], 'msg': 'Input should be greater than 0', 'input': -10, 'ctx': {'gt': 0}}]}


In [ ]:


# ```python
import httpx
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI() # init the fastapi that will allow client and server to talk


@app.get("/github/{username}") # An async endpoint that calls an external public API
async def get_github_stats(username: str):
    # httpx.AsyncClient makes non-blocking HTTP requests across the internet
    async with httpx.AsyncClient() as client:
        response = await client.get(f"https://api.github.com/users/{username}") # why use await here?
    
    if response.status_code != 200:
        return {"error": "User not found on GitHub", "status": response.status_code}
    
    data = response.json()
    # Extract only the fields we care about
    return {
        "user": data.get("login"),
        "public_repos": data.get("public_repos"),
        "followers": data.get("followers")
    }

# Test the endpoint in your notebook
test_client = TestClient(app)

res = test_client.get("/github/torvalds")#,json = {"token_limit":5000})
print("Status:", res.status_code)
print("Data:", res.json())


TypeError: Client.get() got an unexpected keyword argument 'json'

In [ ]:
# ```python
# Continue form here
import sqlite3
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

# 1. Setup local SQLite database
conn = sqlite3.connect("app_data.db", check_same_thread=False) # what is  this connection between?
cursor = conn.cursor() # what is the function of cursor, is it a executer of sql lite conn
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE NOT NULL,
    token_limit INTEGER NOT NULL
)
""")
conn.commit() # what is the meaning of commit here, are we committing anything to the connection?

app = FastAPI()  

# 2. Pydantic request schema
class UserCreate(BaseModel):
    username: str
    token_limit: int = Field(gt=0)

# 3. Endpoints writing to and reading from disk
@app.post("/users")
async def create_user(user: UserCreate):
    try:
        cursor.execute(
            "INSERT INTO users (username, token_limit) VALUES (?, ?)",
            (user.username, user.token_limit)
        ) # cursor makes some changes and then conn commits those to the table?

        conn.commit()
        return {"status": "created", "username": user.username} # what is being returned here, and why are curly bracket used for?
    except sqlite3.IntegrityError:
        raise HTTPException(status_code=400, detail="Username already exists")

@app.get("/users/{username}")
async def get_user(username: str):
    cursor.execute(
        "SELECT id, username, token_limit FROM users WHERE username = ?",
        (username,)
    )# there is no conn.commit means that this query will only extract the rows and will not make any changes to the table
    row = cursor.fetchone()
    if not row:
        raise HTTPException(status_code=404, detail="User not found")
    return {"id": row[0], "username": row[1], "token_limit": row[2]}

# 4. Simulation
client = TestClient(app)

res_create = client.post("/users", json={"username": "alex_dev", "token_limit": 5000}) # how can i remember the parameteres of json, are they standard for each call? or will it come with practice
print("Create response:", res_create.status_code, res_create.json())    

res_get = client.get("/users/alex_dev")
print("Get response:", res_get.json())
# ```

Create response: 400 {'detail': 'Username already exists'}
Get response: [[1, 'alex_dev', 5000]]


In [14]:
from fastapi import FastAPI 
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field 

app = FastAPI()
class ModelConfig(BaseModel): 
    model_name: str 
    temperature: float = Field(ge=0.0, le=1.0) # Must be between 0.0 and 1.0

# BUG 1: The client is going to POST to "/configure", but look at the decorator below 
@app.post("/configure")
async def set_config(config: ModelConfig): 
    return{
        "status": "ready", 
        "model": config.model_name, 
        "temp": config.temperature
    }

client = TestClient(app)

# BUG 2: Look at the JSON payload below. Does temperature satisfy the Field condition? 
res = client.post(
    "/configure", 
    json={"model_name": "gemini-pro", "temperature": 0.8}
)

print("Status:", res.status_code) 
print("Response:", res.json())

Status: 200
Response: {'status': 'ready', 'model': 'gemini-pro', 'temp': 0.8}


In [17]:
import sqlite3 
conn = sqlite3.connect("test_sessions.db") 
cursor = conn.cursor() 

cursor.execute(""" 
CREATE TABLE IF NOT EXISTS sessions (
    id INTEGER PRIMARY KEY AUTOINCREMENT, 
    session_id TEXT UNIQUE NOT NULL, 
    tokens_used INTEGER NOT NULL 
    ) 
    """) 
conn.commit()

# --- YOUR TURN --- 
new_session = ("session_abc123", 150)

# TODO 1: Write the cursor.execute line using ? placeholders to insert new_session 
cursor.execute(
    "INSERT INTO sessions (session_id, tokens_used) VALUES (?, ?)",
            (new_session[0], new_session[1])
)

# TODO 2: Commit the change to disk
conn.commit() 

# Verify: 
cursor.execute("SELECT * FROM sessions WHERE session_id = ?", ("session_abc123",)) 
print("Saved row:", cursor.fetchone()) 
conn.close()

Saved row: (1, 'session_abc123', 150)
